# WikiDT Phase 5 — Standalone OCR → BERT QA (normalized v2)

**Purpose:** run a small, debug-friendly OCR baseline with Amazon Textract annotations from Hugging Face, reusing Google Drive data where possible.

**Corrections:** normalize stringified list answers consistently; exclude `[None]` and missing targets; avoid duplicate question keys across splits; audit full-OCR and top-K answer coverage using the same canonical answer; support repeated answer spans; isolate new outputs in `normalized_v2` to prevent stale checkpoints.

**Start small:** default 100/30/30; change `SIZES` in setup to 24/8/8 for a smoke test, then 450/150/150. Every code cell logs its purpose and elapsed progress. The first OCR archive download may be large.

**X:** question plus retrieved Textract text tokenized for BERT. **Y:** answer start/end token positions; questions without a valid answer span cannot supervise extractive QA.

# Phase 5: WikiDT Amazon Textract → BERT extractive QA

## Research objective
This is a **standalone OCR-based question-answering baseline** for the WikiDT project. It asks: *Given text recognized by Amazon Textract from Wikipedia screenshots, can a pretrained BERT reader locate the correct answer to a natural-language question?* This notebook does **not** train Textract or a ViT/Swin image encoder. It fine-tunes the downstream BERT question-answering model.

### End-to-end workflow
1. Load question/answer metadata from the official Hugging Face WikiDT dataset, optionally reusing a compatible Phase 4 question cohort from Google Drive.
2. Download the official `ocr.tar` archive from Hugging Face and extract only the OCR JSON files referenced by selected questions.
3. Parse Textract `LINE` blocks into text; reuse cached OCR text when available.
4. Split OCR into overlapping chunks and rank them using question-based TF-IDF similarity.
5. Construct BERT inputs **X = (question, OCR context)** and supervision **Y = (answer start token, answer end token)**.
6. Fine-tune a pretrained extractive QA model and measure held-out validation Exact Match (EM) and token F1.

### Pilot size and interpretability
The default cohort is **100 training / 30 validation / 30 test questions**. You can start with `24/8/8` for a fast smoke test. A small pilot tests correctness and integration, **not** generalization. For fair comparison with visual models, eventually use identical question cohorts and scoring.

### Important limitations
- The model can only extract text that appears in its retrieved OCR context; missing or misread OCR text cannot be recovered by an extractive reader.
- Textract `LINE` blocks are flattened into text; this may lose table row/column relationships.
- The current training procedure uses only retrieved windows containing a matched answer. This reduces the number of supervised examples and does not explicitly train the reader to reject irrelevant windows.
- Question-level deduplication does **not** guarantee parent-page-disjoint train/validation/test sets. Section 13 audits page overlap.
- The first download of `ocr.tar` can be large even for a tiny cohort. Google Drive is used for **outputs/caches**, not as the source of Textract OCR.

**How to use this notebook:** Run numbered cells in order. Each cell prints a numbered runtime log; check its counts, sample records, and error messages before continuing.

## 1 — Install dependencies

**Purpose:** Ensure Colab has the Python packages used by the OCR ingestion, text retrieval, tokenization, PyTorch training, and evaluation stages.

**Inputs:** Internet access and the active Colab Python environment. **Outputs:** Installed packages (not dataset artifacts).

**What happens:** This cell disables optional TensorFlow/Flax Transformers backends and installs PyTorch, Hugging Face Hub/Transformers, pandas/Parquet, scikit-learn, and progress-bar dependencies. You should see `[01]` messages before and after installation.

**Debugging:** If an imported Transformers module conflicts with a newly installed version, restart the Colab runtime and run again from cell 1. Package installation alone does not verify that dataset files can be downloaded.

In [ ]:
import os
print('[01] Install packages; run once in Colab')
os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'; os.environ['TRANSFORMERS_NO_TF']='1'
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers>=4.44,<5','huggingface_hub>=0.25,<2','tokenizers>=0.19','torch','pandas','pyarrow','scikit-learn','tqdm'])
print('[01] Packages installed. If Transformers was already imported and fails, restart Colab runtime.')

## 2 — Mount Drive, configure the experiment, and validate answer normalization

**Purpose:** Define a reproducible run, select compute resources, and ensure every stage interprets ground-truth answers consistently.

**Key settings:**
- `SIZES={'train':100,'val':30,'test':30}` controls cohort size. Try `24/8/8` for a quick smoke test; scale to `450/150/150` only after checking retrieval and labels.
- `SEED=42` fixes sampling and random initialization as far as supported by the runtime.
- `MODEL_ID='deepset/bert-base-cased-squad2'` selects the pretrained extractive QA checkpoint.
- `TOP_K=3` selects how many OCR chunks are passed to BERT per question; `CHUNK_WORDS`, `CHUNK_STEP`, `MAX_LEN`, and `DOC_STRIDE` control chunking and tokenizer windows.
- `RUN` is the persistent output directory in Google Drive; `LOCAL` is the temporary Colab cache, which may disappear when the runtime resets.

**Answer normalization:** `answer_text()` unwraps singleton list-style values such as `"['hawthorn']"`, and rejects empty/`None`/`[None]` values. `norm()` is used for comparisons; `eligible()` requires a substantive answer containing at least one letter. This avoids the earlier misleading OCR-coverage measurements.

**Expected output:** `[02]` lines showing device (GPU/CPU), paths, free disk, and passing normalization self-tests.

**Debugging:** If `DEVICE=cpu`, check **Runtime → Change runtime type → GPU**. If Drive mounting fails, check authorization. If you change sizes, confirm the printed `RUN` directory changes accordingly.

In [ ]:
import os, re, json, ast, time, random, tarfile, shutil, unicodedata, hashlib, collections, traceback
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from IPython.display import display
from huggingface_hub import HfApi, hf_hub_download
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
print('[02] Mounting Drive (if running in Colab)')
try:
 from google.colab import drive
 drive.mount('/content/drive')
except ImportError: print('[02] Not in Colab; using local paths')
SIZES={'train':100,'val':30,'test':30} # start with 24/8/8 if preferred; then 450/150/150
SEED=42; EPOCHS=2; BATCH_SIZE=4; LR=2e-5
TOP_K=3; CHUNK_WORDS=170; CHUNK_STEP=120; MAX_LEN=384; DOC_STRIDE=96
MODEL_ID='deepset/bert-base-cased-squad2'; REPO='AmazonScience/WikiDT'; REV='main'
FORCE_RETRAIN=False; USE_PHASE4_COHORT_IF_AVAILABLE=True
ROOT=Path('/content/drive/MyDrive/WikiDT_Persistent') if Path('/content/drive/MyDrive').exists() else Path('/mnt/data/WikiDT_Persistent')
RUN=ROOT/'phase5_ocr_only'/'normalized_v2'/('pilot_'+ '_'.join(str(SIZES[s]) for s in ('train','val','test')))
RUN.mkdir(parents=True,exist_ok=True)
LOCAL=Path('/content/wikidt_phase5_cache') if Path('/content').exists() else Path('/mnt/data/wikidt_phase5_cache')
LOCAL.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
def log(section,msg): print(f'[{section}][{time.strftime("%H:%M:%S")}] {msg}',flush=True)
# One canonical answer conversion shared by sampling, span labeling, audit, and scoring.
MISSING_ANSWERS={'', 'none','no answer','noanswer','nan','null','n/a','[none]','[null]'}
def answer_text(value):
    if value is None: return ''
    if isinstance(value,(list,tuple)):
        return answer_text(value[0]) if len(value)==1 else ''
    if isinstance(value,(float,np.floating)) and np.isnan(value): return ''
    s=str(value).strip()
    if s.startswith('[') and s.endswith(']'):
        try:
            v=ast.literal_eval(s)
            if isinstance(v,(list,tuple)):
                return answer_text(v[0]) if len(v)==1 else ''
        except (ValueError,SyntaxError):
            pass
        if re.fullmatch(r'\[\s*(none|null|nan)\s*\]',s,flags=re.I):return ''
    return '' if s.casefold() in MISSING_ANSWERS else s
def norm(x):
    return ' '.join(unicodedata.normalize('NFKC',answer_text(x)).casefold().split())
def eligible(x):
    a=answer_text(x)
    return bool(a and norm(a) not in MISSING_ANSWERS and any(ch.isalpha() for ch in a))
assert answer_text("['hawthorn']")=='hawthorn'
assert not eligible('[None]') and not eligible('None') and not eligible(['None'])
assert eligible("['tipperary(1950-1959)']")
log('02','Answer normalization self-tests passed; list wrappers and [None] handled')
log('02',f'Device={DEVICE}, sizes={SIZES}, ROOT={ROOT}, RUN={RUN}, local HF cache={LOCAL}')
log('02',f'Free local disk: {shutil.disk_usage(LOCAL).free/1e9:.2f} GB')

## 3 — Inspect the official Hugging Face WikiDT repository

**Purpose:** Discover the **actual** paths to WikiDT sample metadata and Amazon Textract OCR, rather than guessing local file names.

**Input:** Dataset repository `AmazonScience/WikiDT` and revision `main`. **Output:** `SAMPLE_FILES` (train/val/test metadata paths) and `OCR_ARCHIVE` (the `ocr.tar` path).

**What happens:** The Hugging Face API lists repository files; the notebook checks that each required file is found exactly once. This is metadata discovery, **not** the large OCR download.

**Expected output:** Repository file count, example paths, selected metadata paths, and the archive path.

**Debugging:** An error about missing or multiple matches means the upstream repository layout differs from the notebook's assumptions. Inspect the printed repository file list before changing path logic. Network/authentication/rate-limit errors are distinct from path-matching errors.

In [ ]:
t=time.time(); log('03','Listing official Hugging Face dataset files')
api=HfApi(); files=api.list_repo_files(REPO,repo_type='dataset',revision=REV)
log('03',f'Repository has {len(files)} files; first 25: {files[:25]}')
def find_file(suffix):
 matches=[f for f in files if f==suffix or f.endswith('/'+suffix)]
 if len(matches)!=1:raise RuntimeError(f'Expected exactly one {suffix}; matches={matches[:20]}')
 return matches[0]
SAMPLE_FILES={s:find_file('samples/'+s+'.json') for s in SIZES}
OCR_ARCHIVE=find_file('ocr.tar')
log('03',f'Sample files={SAMPLE_FILES}; OCR archive={OCR_ARCHIVE}; elapsed={time.time()-t:.1f}s')

## 4 — Build a small, reproducible QA cohort

**Purpose:** Select the questions to train and evaluate on, using official WikiDT metadata and only questions with associated Textract file references.

**Input:** Hugging Face `samples/{split}.json` metadata, plus optional previously saved Phase 4 cohort Parquet files in Drive.

**What happens:**
1. Parse sample JSON or JSONL records.
2. Normalize the ground-truth answer, reject missing/non-alphabetic targets, and require a nonempty question and `all_ocr_files_textract` references.
3. Try reusing a Phase 4 cohort **only if every selected record uniquely matches official WikiDT metadata and OCR references**; otherwise sample from Hugging Face with a fixed seed.
4. Deduplicate by image basename, normalized question, and normalized answer across splits.
5. Construct `qa`, with `split`, `image`, `question`, `answer`, `ocr_files`, and `cohort_source`.

**Expected output:** Raw and selected counts for each split, data source, and unique OCR reference counts. Review a few rows to ensure the questions, answers, and OCR paths make sense.

**Debugging:** If fewer eligible rows than requested are available, reduce `SIZES` or inspect metadata fields. **Caution:** This is not a guaranteed page-disjoint split; the later diagnostics report page overlap.

In [ ]:
t=time.time(); log('04','Loading HF sample metadata (not OCR archive)')
def records(path):
 raw=Path(path).read_text(encoding='utf-8')
 try: obj=json.loads(raw)
 except json.JSONDecodeError:obj=[json.loads(line) for line in raw.splitlines() if line.strip()]
 if isinstance(obj,list):return obj
 if isinstance(obj,dict):
  for k in ('samples','data','records','items'):
   if isinstance(obj.get(k),list):return obj[k]
  if all(isinstance(v,dict) for v in obj.values()):return list(obj.values())
 raise ValueError('Unsupported sample JSON structure: '+str(type(obj)))
def safe_ref(x):
 p=str(x).replace('\\','/').lstrip('./')
 if 'ocr/textract/' in p:p='ocr/textract/'+p.split('ocr/textract/',1)[1]
 if '..' in Path(p).parts:raise ValueError('Unsafe path '+p)
 return p
def refs(r):
 v=r.get('all_ocr_files_textract') or []
 if isinstance(v,str):v=[v]
 return sorted(set(safe_ref(x) for x in v if isinstance(x,str) and 'textract/' in x.replace('\\','/')))
def key(r):return (Path(str(r.get('image',''))).name.casefold(),norm(r.get('question','')),norm(answer_text(r.get('answer',''))))
manifest=[]
used_keys=set()
for split,count in SIZES.items():
 path=hf_hub_download(repo_id=REPO,repo_type='dataset',filename=SAMPLE_FILES[split],revision=REV,cache_dir=str(LOCAL))
 raw=[r for r in records(path) if isinstance(r,dict)]
 log('04',f'{split}: HF records={len(raw)}; sample keys={list(raw[0])[:20] if raw else []}')
 idx=collections.defaultdict(list)
 for r in raw:idx[key(r)].append(r)
 saved_candidates=[ROOT/'embeddings'/'vit_bert_qa_baseline'/'phase4_alpha_mixed_v1_450_150_150'/f'{split}_fixed_qa.parquet',ROOT/'embeddings'/'vit_bert_qa_baseline'/'phase4_alpha_only_v1_450_150_150'/f'{split}_fixed_qa.parquet']
 chosen=[];source='HF fresh'
 if USE_PHASE4_COHORT_IF_AVAILABLE:
  for cp in saved_candidates:
   if cp.exists():
    old=pd.read_parquet(cp)
    if len(old)>=count:
     trial=old.sample(n=count,random_state=SEED) if len(old)>count else old
     trial_keys=[(Path(str(r.image)).name.casefold(),norm(r.question),norm(answer_text(r.answer))) for r in trial.itertuples(index=False)]
     if all(len(idx[k])==1 and refs(idx[k][0]) and eligible(idx[k][0].get('answer')) for k in trial_keys):
      chosen=[idx[k][0] for k in trial_keys] if not any(k in used_keys for k in trial_keys) else [];source=f'Drive Phase4 {cp.name}'
      break
     log('04',f'{split}: existing Phase4 rows not all uniquely matched to HF OCR refs; fallback to HF')
 if not chosen:
  pool=[];seen=set()
  for r in raw:
   k=key(r); a=answer_text(r.get('answer',''))
   if k in seen or k in used_keys or not eligible(a) or not str(r.get('question','')).strip() or not refs(r):continue
   seen.add(k);pool.append(r)
  if len(pool)<count:raise RuntimeError(f'{split}: only {len(pool)} eligible HF records with OCR refs; requested {count}')
  chosen=random.Random(SEED+{'train':0,'val':1,'test':2}[split]).sample(pool,count)
 for r in chosen:
  if key(r) in used_keys: raise RuntimeError('Overlapping sample key across splits')
  used_keys.add(key(r))
  manifest.append(dict(split=split,image=str(r.get('image','')),question=str(r.get('question','')),answer=answer_text(r.get('answer','')),ocr_files=refs(r),cohort_source=source))
 log('04',f'{split}: selected {len(chosen)} questions, source={source}, unique OCR refs={len(set(p for r in chosen for p in refs(r)))}')
qa=pd.DataFrame(manifest)
assert qa.answer.map(eligible).all(), 'Missing or numeric-only answer slipped into cohort'
log('04',f'Canonical answer checks passed: {len(qa)} eligible questions')
qa.assign(ocr_files=qa.ocr_files.map(json.dumps)).to_parquet(RUN/'qa_manifest.parquet',index=False)
log('04',f'Saved manifest; elapsed={time.time()-t:.1f}s')
display(qa[['split','question','answer','cohort_source']].head(5))

## 5 — Download the official Textract OCR archive

**Purpose:** Obtain the **existing Amazon Textract output** from Hugging Face. No OCR model is trained or invoked here.

**Input:** All distinct `ocr_files` paths from `qa` and the discovered Hugging Face `ocr.tar` file.

**What happens:** Query the archive's published size, check local free disk, then download/cache the archive with `hf_hub_download`. This is often the longest initial step. A small pilot reduces the number of JSONs extracted, **not necessarily the archive download size**.

**Expected output:** Number of needed OCR JSON paths, archive size if available, free disk space, cached archive path, elapsed time.

**Debugging:** If disk is insufficient, increase Colab storage or change how the archive is streamed; changing `SIZES` alone will not shrink `ocr.tar`. A Hugging Face download error should be investigated separately from training issues. Drive is intentionally not searched as an OCR source.

In [ ]:
t=time.time(); wanted=sorted({p for arr in qa.ocr_files for p in arr})
log('05',f'Need {len(wanted)} unique OCR JSON paths for this pilot')
info=api.repo_info(repo_id=REPO,repo_type='dataset',revision=REV,files_metadata=True)
entry=next((x for x in info.siblings if x.rfilename==OCR_ARCHIVE),None)
size=getattr(entry,'size',None)
free=shutil.disk_usage(LOCAL).free
log('05',f'ocr.tar published size={size/1e9:.2f} GB' if size else 'ocr.tar size unavailable')
log('05',f'Free local disk={free/1e9:.2f} GB')
if size and free < size*1.10:
 raise RuntimeError('Insufficient local disk for Hugging Face ocr.tar. Increase Colab storage; this notebook intentionally does not search Drive for OCR.')
archive=Path(hf_hub_download(repo_id=REPO,repo_type='dataset',filename=OCR_ARCHIVE,revision=REV,cache_dir=str(LOCAL)))
log('05',f'Archive downloaded/cached: {archive}; elapsed={time.time()-t:.1f}s')

## 6 — Match and extract the selected OCR JSON files

**Purpose:** Map the sample metadata's OCR references to exact TAR members, without extracting the full archive.

**Input:** Downloaded `ocr.tar` and the set of referenced OCR paths. **Output:** `local_paths`, mapping each official OCR reference to a local JSON file.

**What happens:** Canonicalize paths, scan TAR members, flag ambiguous matches, and copy only matched JSON streams into a hashed local filename. Hashing avoids unsafe paths and name collisions.

**Expected output:** Counts of exact matches, missing paths, ambiguous paths, and extraction progress.

**Debugging:** If many paths are missing, compare sample `all_ocr_files_textract` references with TAR member names. Do not silently substitute a similarly named JSON file; that could attach the wrong table to a question. Missing files can lower the OCR coverage ceiling.

In [ ]:
t=time.time(); log('06','Scanning TAR members and selecting exact OCR paths')
ocr_dir=LOCAL/'selected_textract';ocr_dir.mkdir(exist_ok=True)
def canon(x):
 x=str(x).replace('\\','/').lstrip('./')
 return x.split('ocr/textract/',1)[-1] if 'ocr/textract/' in x else x
needed={canon(x):x for x in wanted}
member_map={}; ambiguous=set()
with tarfile.open(archive,'r:*') as tar:
 for member in tqdm(tar,desc='Scan TAR'):
  if not member.isfile():continue
  c=canon(member.name)
  if c in needed:
   if c in member_map:ambiguous.add(c)
   else:member_map[c]=member
 for c in ambiguous:member_map.pop(c,None)
 missing=sorted(set(needed)-set(member_map))
 log('06',f'Exact matched={len(member_map)}, missing={len(missing)}, ambiguous={len(ambiguous)}')
 if missing:log('06',f'First missing refs: {missing[:10]}')
 if ambiguous:log('06',f'First ambiguous refs: {list(ambiguous)[:10]}')
 if not member_map:raise RuntimeError('No OCR paths matched archive. Inspect sample OCR refs and TAR member names; do not guess.')
 for c,m in tqdm(member_map.items(),desc='Extract selected OCR'):
  dest=ocr_dir/hashlib.sha256(c.encode()).hexdigest()[:24]
  if dest.exists() and dest.stat().st_size:continue
  stream=tar.extractfile(m)
  if stream is None:continue
  with open(dest,'wb') as f:shutil.copyfileobj(stream,f)
local_paths={needed[c]:ocr_dir/hashlib.sha256(c.encode()).hexdigest()[:24] for c in member_map}
log('06',f'Selected OCR files on local disk={len(local_paths)}; elapsed={time.time()-t:.1f}s')
example=next(iter(local_paths.values()))
print('[06] Sample OCR JSON prefix:',example.read_text(encoding='utf-8',errors='replace')[:700])

## 7 — Parse Textract into readable text and audit OCR coverage

**Purpose:** Convert Textract JSON annotations into a text corpus for each question and reuse any previously parsed OCR text.

**Input:** Extracted OCR JSONs and existing Drive `ocr_text_cache.parquet`, if present. **Output:** Text mapped to OCR references, plus question-level OCR availability and answer-coverage diagnostics.

**What happens:** Prefer Textract `LINE` blocks (rather than indiscriminately concatenating `WORD` and `LINE` blocks, which would duplicate text). Assemble the OCR text associated with each selected question. Check whether the normalized ground-truth answer appears anywhere in its OCR text.

**Expected output:** Number of OCR documents parsed or reused, examples of recognized text, and coverage statistics.

**Interpretation:** **OCR answer coverage** measures whether the answer string is present in available recognized text. It is an upper bound for a purely extractive model using that text; it is not the accuracy of BERT. Exact string coverage may undercount recoverable cases when punctuation or spacing differs.

**Debugging:** If coverage is unexpectedly low, inspect raw JSON structure, OCR paths, and answer normalization before training. Flattened `LINE` text may lose table geometry.

In [ ]:
t=time.time();log('07','Parsing Textract JSON; reusing Drive text cache')
cache=RUN/'ocr_text_cache.parquet'
text_map={}
if cache.exists():
 old=pd.read_parquet(cache)
 text_map=dict(zip(old.ocr_path.astype(str),old.text.astype(str)))
 log('07',f'Loaded {len(text_map)} cached OCR texts')
def textract_text(obj):
 if isinstance(obj,list):
  # Support the custom compact list format seen in cell 6
  # structure: [ [page_num, [ [text, [coords]], ... ]], ... ]
  lines=[]
  for item in obj:
   if isinstance(item,list) and len(item)==2 and isinstance(item[1],list):
    for text_block in item[1]:
     if isinstance(text_block,list) and len(text_block)>=1 and isinstance(text_block[0],str):
      if text_block[0].strip():
       lines.append(text_block[0])
  if lines:
   return '\n'.join(lines)
  return '\n'.join(x for y in obj if (x:=textract_text(y)))
 if isinstance(obj,dict):
  blocks=obj.get('Blocks',obj.get('blocks'))
  if isinstance(blocks,list):
   lines=[str(b.get('Text','')) for b in blocks if isinstance(b,dict) and str(b.get('BlockType','')).upper()=='LINE' and b.get('Text')]
   if lines:return '\n'.join(lines)
   words=[str(b.get('Text','')) for b in blocks if isinstance(b,dict) and str(b.get('BlockType','')).upper()=='WORD' and b.get('Text')]
   if words:return ' '.join(words)
  for k in ('textract','response','result','data'):
   if isinstance(obj.get(k),(dict,list)):
    v=textract_text(obj[k])
    if v:return v
  if isinstance(obj.get('text'),str):return obj['text']
 return ''
for p,loc in tqdm(local_paths.items(),desc='Parse OCR'):
 if p in text_map and text_map[p].strip():continue
 try:text_map[p]=textract_text(json.loads(loc.read_text(encoding='utf-8')))
 except Exception as e:log('07',f'ERROR parsing {p}: {e}');text_map[p]=''
pd.DataFrame([{'ocr_path':k,'text':v} for k,v in text_map.items()]).to_parquet(cache,index=False)
log('07',f'Nonempty={sum(bool(v.strip()) for v in text_map.values())}/{len(wanted)} requested; elapsed={time.time()-t:.1f}s')
print('[07] Example OCR TEXT (X context source):',repr(next((v[:600] for v in text_map.values() if v.strip()),'NONE')))
if not any(v.strip() for v in text_map.values()):raise RuntimeError('OCR parser produced no text; inspect JSON sample in cell 6.')

## 8 — Build question-aware OCR retrieval and inspect X/Y in plain text

**Purpose:** Reduce long Wikipedia OCR documents into manageable passages likely to contain the answer.

**Inputs:** Question `Q`, all associated OCR text, and the gold answer **only for coverage auditing**.

**Procedure:** Split OCR text into overlapping word chunks, compute TF-IDF similarity between the question and candidate chunks, and keep the top `TOP_K` chunks. Retrieval ranking is **answer-blind**: the answer must not be used to choose the chunk.

**Intermediate model input X:** `(question, retrieved OCR chunk)`.

**Intermediate label Y:** The gold answer string, used later to locate character and token spans in the retrieved context. The example printed in this cell shows the question, a selected OCR chunk, the original answer, and any matched character span.

**Key metrics:**
- **OCR coverage:** answer found anywhere in the available OCR.
- **Top-K retrieval coverage:** answer found in at least one retrieved chunk.

**Debugging:** High OCR coverage but low top-K coverage means retrieval is the bottleneck. Try larger `TOP_K`, better chunking, or layout-aware table serialization. Do not confuse retrieval failure with BERT prediction failure.

In [ ]:
t=time.time();log('08','Creating answer-blind TF-IDF retrieval contexts')
def chunks(txt):
 w=str(txt).split();return [' '.join(w[i:i+CHUNK_WORDS]) for i in range(0,len(w),CHUNK_STEP) if w[i:i+CHUNK_WORDS]]
def span(context,answer):
 m=re.search(re.escape(str(answer).strip()),context,flags=re.I)
 return (m.start(),m.end()) if m else None
def retrieve(q,paths):
 candidates=[{'path':p,'context':c} for p in paths for c in chunks(text_map.get(p,''))]
 if not candidates:return []
 try:
  tf=TfidfVectorizer(ngram_range=(1,2),stop_words='english'); m=tf.fit_transform([q]+[r['context'] for r in candidates]); scores=cosine_similarity(m[:1],m[1:]).ravel()
 except ValueError:scores=np.zeros(len(candidates))
 return [dict(candidates[i],score=float(scores[i])) for i in np.argsort(-scores,kind='stable')[:TOP_K]]
rows=[]
assert qa.answer.map(eligible).all(), 'Invalid answer found before retrieval'
for r in tqdm(qa.itertuples(index=False),total=len(qa),desc='Retrieve'):
 hits=retrieve(r.question,r.ocr_files)
 alltext='\\n'.join(text_map.get(p,'') for p in r.ocr_files)
 rows.append(dict(split=r.split,image=r.image,question=r.question,answer=answer_text(r.answer),hits_json=json.dumps(hits),ocr_present=bool(alltext.strip()),answer_in_ocr=span(alltext,answer_text(r.answer))!=None,answer_in_topk=any(span(h['context'],answer_text(r.answer))!=None for h in hits)))
audit=pd.DataFrame(rows)
assert audit.answer.map(eligible).all(), 'Invalid answer in audit'
assert not (audit.answer_in_topk & ~audit.answer_in_ocr).any(), 'Top-K coverage cannot exceed full OCR coverage'
audit.to_parquet(RUN/'retrieval_audit.parquet',index=False)
for s,d in audit.groupby('split'):
 log('08',f'{s}: N={len(d)}, OCR present={d.ocr_present.mean():.1%}, answer in OCR={d.answer_in_ocr.mean():.1%}, answer in topK={d.answer_in_topk.mean():.1%}')
r=audit.iloc[0];hs=json.loads(r.hits_json)
print('\\n[08] EXAMPLE BEFORE TOKENIZATION')
print('X.question:',r.question,'\\nX.context:',hs[0]['context'][:650] if hs else '<NO OCR>')
print('Y.answer string:',repr(r.answer),'\\nY.character span:',span(hs[0]['context'],r.answer) if hs else None)
log('08',f'Elapsed {time.time()-t:.1f}s')

## 9 — Convert X and Y into supervised BERT training features

**Purpose:** Transform human-readable question/context/answer examples into tokenized tensors and token-position labels.

### X: model features
For each retrieved passage, the fast tokenizer constructs a paired input of the form:

`[CLS] question tokens [SEP] OCR context tokens [SEP]`

The feature dictionary includes `input_ids` (token IDs), `attention_mask` (real tokens vs padding), and possibly `token_type_ids` (question vs context segment IDs). The tokenizer also returns character `offset_mapping` so token positions can be mapped back to the OCR text.

### Y: supervised targets
The answer string is searched in the OCR context; valid occurrences are converted into **`start_positions`** and **`end_positions`** token indices. These are the two labels optimized by the QA head. For example, if the context contains `Glenferrie Oval Hawthorn`, the label identifies the token span for `Hawthorn`—**not** a class ID for the word.

### Filtering and debugging
Only windows with a verified answer span are used for supervised training. The cell prints a **matched X/Y example**: question, context, first tokens and IDs, attention mask, gold answer, token indices, and text reconstructed from the token offsets. It asserts that the reconstructed answer matches the target.

**Expected output:** Number of training questions with usable spans, total supervised windows, and a few uncovered examples.

**Interpretation:** If 100 questions yield only 78 usable spans, training effectively uses 78 questions; the remaining 22 are not trained as negative examples in this implementation. A low usable-span count calls for OCR/retrieval improvements before increasing epochs.

In [ ]:
t=time.time();log('09','Loading fast BERT tokenizer and constructing span labels')
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
assert tokenizer.is_fast,'A fast tokenizer is required for offset mappings'
def features(row,training=False):
 result=[]
 for h in json.loads(row.hits_json):
  context=h['context']
  ans_to_match = answer_text(row.answer)
  golds=[(m.start(),m.end()) for m in re.finditer(re.escape(ans_to_match),context,flags=re.I)] if training and ans_to_match else []
  if training and not golds:continue
  enc=tokenizer(str(row.question),context,max_length=MAX_LEN,truncation='only_second',stride=DOC_STRIDE,return_overflowing_tokens=True,return_offsets_mapping=True,padding='max_length')
  for j in range(len(enc['input_ids'])):
   offsets=enc['offset_mapping'][j]; seq=enc.sequence_ids(j)
   valid=[i for i,s in enumerate(seq) if s==1 and offsets[i][1]>offsets[i][0]]
   if not valid:continue
   rec={'input_ids':enc['input_ids'][j],'attention_mask':enc['attention_mask'][j],'offsets':offsets,'sequence_ids':seq,'context':context,'score':h['score']}
   if 'token_type_ids' in enc:rec['token_type_ids']=enc['token_type_ids'][j]
   if training:
    gold=next((g for g in golds if offsets[valid[0]][0]<=g[0] and offsets[valid[-1]][1]>=g[1]),None)
    if gold is None:continue
    start=next((i for i in valid if offsets[i][0]<=gold[0]<offsets[i][1]),None)
    end=next((i for i in reversed(valid) if offsets[i][0]<gold[1]<=offsets[i][1]),None)
    if start is None or end is None:continue
    rec['start_positions']=start;rec['end_positions']=end
   result.append(rec)
 return result
train=[];covered=0;example_row=None;uncovered_examples=[]
for r in tqdm(audit[audit.split=='train'].itertuples(index=False),desc='Build train features'):
 f=features(r,training=True)
 if f:
  covered+=1;train.extend(f)
  if example_row is None: example_row=r
 else: uncovered_examples.append({'question':r.question,'answer':r.answer,'answer_in_ocr':r.answer_in_ocr,'answer_in_topk':r.answer_in_topk})
log('09',f'Training questions with valid token span={covered}/{SIZES["train"]}; supervised windows={len(train)}')
if train:
 ex=train[0];print("\n[09] EXAMPLE MODEL TRAINING TENSORS")
 print('X.question:',example_row.question)
 print('X.context:',ex['context'][:450])
 print('X.input_ids[:35]:',ex['input_ids'][:35]);print('X.tokens[:35]:',tokenizer.convert_ids_to_tokens(ex['input_ids'][:35]))
 print('X.attention_mask[:35]:',ex['attention_mask'][:35])
 print('Y.start_positions:',ex['start_positions'],'Y.end_positions:',ex['end_positions'])
 print('Y.ground_truth:',repr(example_row.answer))
 extracted=ex['context'][ex['offsets'][ex['start_positions']][0]:ex['offsets'][ex['end_positions']][1]]
 print('Y.extracted_text:',repr(extracted))
 assert extracted.casefold()==example_row.answer.casefold(), 'X/Y example does not match!'
 print('[09] Verified printed X and Y belong to the same training question')
else:
 raise RuntimeError('No supervised answer spans in top-K OCR. Inspect cells 6-8, increase pilot sample or TOP_K, or improve retrieval before training.')
print('[09] Missing training span examples:',uncovered_examples[:8])
log('09',f'Elapsed {time.time()-t:.1f}s')

## 10 — Fine-tune the pretrained BERT extractive reader

**Purpose:** Teach the QA model to identify answer boundaries in OCR passages, using the X/Y tensors from Section 9.

**Trainable components:** The pretrained BERT encoder and its start/end QA prediction head. **Not trained:** Amazon Textract, the TF-IDF retriever, or any visual encoder.

**Loss:** The model predicts a start-logit and end-logit for each input token. Training minimizes the average of start-position and end-position cross-entropy losses:

\[
\mathcal{L}=	frac12\left(\mathcal{L}_{start}+\mathcal{L}_{end}ight).
\]

**Configuration:** `EPOCHS`, `BATCH_SIZE`, and `LR` are defined in Section 2. The cell prints the device, number of batches, batch-level losses, epoch-average loss, and elapsed time. It saves a checkpoint and training history in `RUN`.

**Checkpoint safety:** A fingerprint of the selected questions and relevant training settings is stored beside the checkpoint. If the fingerprint changes, the notebook retrains rather than silently evaluating an incompatible older model. Set `FORCE_RETRAIN=True` to override reuse.

**Debugging:** A falling training loss is expected but does **not** prove generalization. If loss becomes `NaN`, inspect token labels, learning rate, and input lengths. If you run out of GPU memory, reduce `BATCH_SIZE` or `MAX_LEN`.

In [ ]:
t=time.time();log('10',f'Start training, device={DEVICE}, epochs={EPOCHS}, batches/epoch={(len(train)+BATCH_SIZE-1)//BATCH_SIZE}')
class TrainSet(Dataset):
 def __len__(self):return len(train)
 def __getitem__(self,i):
  r=train[i];return {k:torch.tensor(r[k],dtype=torch.long) for k in ('input_ids','attention_mask','start_positions','end_positions','token_type_ids') if k in r}
loader=DataLoader(TrainSet(),batch_size=BATCH_SIZE,shuffle=True)
ckpt=RUN/'bert_qa_checkpoint'
# Training-data fingerprint prevents silently loading a checkpoint trained on different samples/settings.
training_signature=hashlib.sha256(json.dumps({'manifest':qa[['split','image','question','answer']].to_dict('records'),'model':MODEL_ID,'epochs':EPOCHS,'top_k':TOP_K,'chunk_words':CHUNK_WORDS,'max_len':MAX_LEN,'lr':LR},sort_keys=True,default=str).encode()).hexdigest()
meta_file=ckpt/'training_signature.json'
checkpoint_ok=(ckpt/'config.json').exists() and meta_file.exists() and json.loads(meta_file.read_text()).get('sha256')==training_signature
if (ckpt/'config.json').exists() and not checkpoint_ok:
 log('10','Existing checkpoint fingerprint differs or is absent; retraining to avoid stale-model evaluation')
if checkpoint_ok and not FORCE_RETRAIN:
 model=AutoModelForQuestionAnswering.from_pretrained(str(ckpt)).to(DEVICE)
 log('10',f'Reused checkpoint {ckpt}')
else:
 model=AutoModelForQuestionAnswering.from_pretrained(MODEL_ID).to(DEVICE)
 opt=AdamW(model.parameters(),lr=LR)
 history=[]
 for ep in range(EPOCHS):
  model.train();losses=[];start=time.time()
  for step,batch in enumerate(tqdm(loader,desc=f'Epoch {ep+1}/{EPOCHS}'),1):
   batch={k:v.to(DEVICE) for k,v in batch.items()}
   opt.zero_grad(set_to_none=True);loss=model(**batch).loss
   loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
   losses.append(float(loss.detach().cpu()))
   if step==1 or step%10==0:log('10',f'Epoch {ep+1}, batch {step}/{len(loader)}, loss={losses[-1]:.4f}')
  mean=float(np.mean(losses));history.append(dict(epoch=ep+1,loss=mean,seconds=time.time()-start))
  log('10',f'Epoch {ep+1} complete: loss={mean:.4f}, seconds={time.time()-start:.1f}')
 ckpt.mkdir(parents=True,exist_ok=True);model.save_pretrained(str(ckpt));tokenizer.save_pretrained(str(ckpt));meta_file.write_text(json.dumps({'sha256':training_signature,'model':MODEL_ID,'sizes':SIZES,'epochs':EPOCHS},indent=2))
 pd.DataFrame(history).to_csv(RUN/'train_history.csv',index=False)
 log('10',f'Saved model {ckpt}')
log('10',f'Training cell elapsed={time.time()-t:.1f}s')

## 11 — Evaluate the full held-out validation cohort

**Purpose:** Measure whether the trained reader extracts correct answers from retrieved Textract text for **all** validation questions, not just questions with an answer-containing training window.

**Inference:** For each question, tokenize each retrieved passage (including overflow windows), run BERT, and choose the highest-scoring valid start/end token span within the configured answer-length limit. The predicted text is sliced from the OCR context using tokenizer offsets.

**Metrics:**
- **Exact Match (EM):** normalized predicted answer exactly equals normalized gold answer.
- **Token F1:** overlap of normalized predicted and gold answer tokens, allowing partial credit.
- **OCR answer coverage:** gold answer occurs somewhere in the associated OCR.
- **Top-K retrieval coverage:** gold answer occurs in one of the selected OCR chunks.
- **Conditional EM:** EM on questions for which the retrieved context contains the gold answer.

**Expected output:** A summary line with counts and percentages, a table showing `question`, `answer`, `prediction`, `EM`, `F1`, and coverage flags, and `val_predictions.parquet` saved to Drive.

**Interpretation:** If retrieval coverage is high but EM is low, investigate the QA reader and table serialization. With 30 validation questions, each exact match changes EM by about 3.33 percentage points; do not overinterpret small changes. The test split is intentionally left untouched until model choices are finalized.

In [ ]:
t=time.time();log('11','Evaluating full validation cohort')
def predict(r,max_answer_tokens=25):
 fs=features(r,training=False)
 if not fs:return '',None
 best=(-float('inf'),'');model.eval()
 with torch.no_grad():
  for f in fs:
   x={k:torch.tensor([f[k]],device=DEVICE) for k in ('input_ids','attention_mask','token_type_ids') if k in f}
   out=model(**x)
   a=out.start_logits[0].cpu().numpy();b=out.end_logits[0].cpu().numpy()
   valid=[i for i,s in enumerate(f['sequence_ids']) if s==1 and f['offsets'][i][1]>f['offsets'][i][0]]
   for pos,i in enumerate(valid):
    for j in valid[pos:pos+max_answer_tokens]:
     lo,hi=f['offsets'][i][0],f['offsets'][j][1]
     candidate=f['context'][lo:hi].strip();score=float(a[i]+b[j])
     if candidate and score>best[0]:best=(score,candidate)
 return best[1],(best[0] if np.isfinite(best[0]) else None)
def token_f1(p,g):
 a=norm(p).split();b=norm(g).split()
 if not a or not b:return float(a==b)
 overlap=sum((collections.Counter(a)&collections.Counter(b)).values())
 return 2*overlap/(len(a)+len(b))
def evaluate(split='val'):
 d=audit[audit.split==split].copy()
 predictions=[predict(r) for r in tqdm(d.itertuples(index=False),total=len(d),desc='Evaluate '+split)]
 d['prediction']=[p for p,s in predictions];d['score']=[s for p,s in predictions]
 d['EM']=[norm(p)==norm(g) for p,g in zip(d.prediction,d.answer)]
 d['F1']=[token_f1(p,g) for p,g in zip(d.prediction,d.answer)]
 log('11',f'{split}: N={len(d)}, correct={int(d.EM.sum())}, EM={d.EM.mean():.2%}, F1={d.F1.mean():.2%}, OCR answer coverage={d.answer_in_ocr.mean():.2%}, topK answer coverage={d.answer_in_topk.mean():.2%}')
 subset=d[d.answer_in_topk]
 log('11',f'EM conditional on answer in retrieved context={subset.EM.mean():.2%}' if len(subset) else 'No answer-containing retrieved validation contexts')
 d.to_parquet(RUN/f'{split}_predictions.parquet',index=False)
 display(d[['question','answer','prediction','EM','F1','answer_in_ocr','answer_in_topk']].head(15))
 return d
validation=evaluate('val')
log('11',f'Elapsed {time.time()-t:.1f}s')

## 12 — Inspect validation failures and plan the next run

**Purpose:** Separate errors caused by missing OCR, incomplete retrieval, and incorrect QA extraction.

**Failure groups:**
1. **Missing OCR:** No usable OCR text for the question.
2. **Answer absent from OCR:** OCR exists, but the answer string cannot be found.
3. **Retrieval missed answer:** Answer is in OCR, but not in the top-K passages.
4. **QA wrong despite retrieved answer:** The answer is available to BERT, but the predicted span is incorrect.

The cell prints counts and a few representative question/answer/prediction rows for each category. It also shows the output directory and the optional held-out test evaluation command.

**Debugging decision:** Improve file matching/Textract parsing for groups 1–2, retrieval and chunking for group 3, and model training/context formatting for group 4. These groups are intended as a diagnostic decomposition, not a substitute for manually inspecting failures.

In [ ]:
log('12','Failure breakdown and next experiment guidance')
v=validation
for name,mask in [('Missing OCR',~v.ocr_present),('Answer absent from OCR',v.ocr_present & ~v.answer_in_ocr),('Retrieval missed answer',v.answer_in_ocr & ~v.answer_in_topk),('QA wrong despite retrieved answer',v.answer_in_topk & ~v.EM)]:
 print(f'{name}: {int(mask.sum())}/{len(v)}')
 display(v.loc[mask,['question','answer','prediction']].head(3))
print('\\nArtifacts saved to:',RUN)
print('Next sizes: 100/30/30, then 450/150/150. Re-run top-to-bottom after changing SIZES.')
print('Optional held-out evaluation after model selection: test_predictions = evaluate("test")')

## 13 — Audit leakage, training span availability, and reader errors

**Purpose:** Perform final quality checks before scaling up or reporting results in the CS 266 project.

**Checks:**
- Count shared **parent-image basenames** between train, validation, and test. A nonzero overlap can make validation optimistic because multiple questions may refer to the same Wikipedia page.
- Report how many training questions had a verified supervised token span.
- Reprint validation OCR coverage, top-K retrieval coverage, and EM conditional on successful retrieval.
- Display examples where the gold answer was retrieved but BERT selected the wrong text.

**Expected output:** `[13]` diagnostic lines and a small failure table.

**Research reporting:** Record cohort sizes, number of supervised questions/windows, OCR and retrieval coverage, EM/F1, and parent-page overlap. For a stronger final comparison with ViT/Swin, construct page-disjoint splits and keep the evaluation cohort fixed across methods.

**Next steps:** After a clean 100/30/30 run, consider 450/150/150 and improve table-aware OCR serialization; do not scale solely because training loss decreases.

In [ ]:
log('13','Diagnostics: leakage, span coverage, and reader failures')
for left,right in [('train','val'),('train','test'),('val','test')]:
    a=set(qa.loc[qa.split==left,'image'].map(lambda p: Path(str(p)).name.casefold()))
    b=set(qa.loc[qa.split==right,'image'].map(lambda p: Path(str(p)).name.casefold()))
    log('13',f'Parent image overlap {left}/{right}: {len(a & b)}; example IDs={list(sorted(a & b))[:5]}')
log('13',f'Train usable answer spans: {covered}/{SIZES["train"]}; missing={len(uncovered_examples)}')
log('13',f'Validation OCR coverage: {validation.answer_in_ocr.mean():.2%}')
log('13',f'Validation top-{TOP_K} retrieval coverage: {validation.answer_in_topk.mean():.2%}')
log('13',f'Validation reader EM given retrieved answer: {validation.loc[validation.answer_in_topk,"EM"].mean():.2%}')
log('13','Failures where answer is available in retrieved context:')
display(validation.loc[validation.answer_in_topk & ~validation.EM,['question','answer','prediction','F1']].head(12))
print('[13] Next steps: inspect OCR table ordering, improve retrieval and increase training samples; do not interpret training-loss reduction alone as generalization.')